In [ ]:
import pandas as pd 
import numpy as np 
import os
DATA_DIR = "data"

In [30]:
col = ['Time' , 'Channel1','Channel2','Channel3','Channel4','Channel5','Channel6',
       'Channel7','Channel8','Class']
df = pd.read_csv('gesture1.txt',sep=r'\s+')
df['class'].value_counts().reset_index()



,class,count
0,0,33781
1,6,3857
2,5,3423
3,4,3218
4,2,3188
5,3,3072
6,1,3014


In [31]:
df.head()

,time,channel1,channel2,channel3,channel4,channel5,channel6,channel7,channel8,class
0,21,-0.00001,0.00003,-0.00003,0.00001,0.00004,0.00001,-0.00002,-0.00004,0
1,57,-0.00001,-0.00001,-0.00004,-0.00005,-0.00004,-0.00003,-0.00004,-0.00002,0
2,58,-0.00002,-0.00001,-0.00003,-0.00003,-0.00003,0.00002,-0.00002,0.00000,0
3,59,-0.00002,-0.00001,-0.00003,-0.00003,-0.00003,0.00002,-0.00002,0.00000,0
4,60,-0.00002,-0.00001,-0.00003,-0.00003,-0.00003,0.00002,-0.00002,0.00000,0


Here i need to check the time difference btween each samples
so , i can generate a sampling rate 

In [32]:
df['time'].diff().value_counts()
print(len(df))

53553


This is the function to extract important time domain and frequency domain features from each channel

In [33]:
def extract_features(data):
    features_list = []
    window_size = 200
    sampling_rate = 1000  # Hz, based on your 1ms median sampling interval

    for start in range(0, len(data) - window_size, window_size):
        window = data.iloc[start: start + window_size]
        window_class = window['class'].mode()[0]
        row = {}

        for ch in ['channel1','channel2','channel3','channel4','channel5','channel6','channel7','channel8']:
            signal = window[ch].values

            # Time-domain features (already have these)
            rms = np.sqrt(np.mean(signal**2))
            mav = np.mean(np.abs(signal))

            # Frequency-domain feature (new)
            fft_vals = np.fft.fft(signal)
            fft_freqs = np.fft.fftfreq(len(signal), d=1/sampling_rate)
            magnitude = np.abs(fft_vals)

            # only keep first half (FFT output is mirrored)
            half = len(signal) // 2
            freqs_half = fft_freqs[:half]
            mag_half = magnitude[:half]

            # mean frequency = weighted average frequency, weighted by magnitude
            mean_freq = np.sum(freqs_half * mag_half) / np.sum(mag_half)

            row[f'{ch}_rms'] = rms
            row[f'{ch}_mav'] = mav
            row[f'{ch}_meanfreq'] = mean_freq

        row['class'] = window_class
        features_list.append(row)

    features_df = pd.DataFrame(features_list)
    return features_df
dfx = pd.read_csv('gesture1.txt',sep=r'\s+')
df_1 = extract_features(dfx)
df_1.shape

(267, 25)

In [34]:
gesture_list = []
for i in range(1, 73):
    gesture_list.append(f'gesture{i}.txt')


In [ ]:
all_features = []
for i,ges in enumerate(gesture_list, start=1):
    dat = pd.read_csv(os.path.join(DATA_DIR,ges), sep=r'\s+')
    feat = extract_features(dat)
    feat['Subject_id'] = i
    all_features.append(feat)

final_df = pd.concat(all_features, ignore_index=True)
final_df.head()


,channel1_rms,channel1_mav,channel1_meanfreq,channel2_rms,channel2_mav,channel2_meanfreq,channel3_rms,channel3_mav,channel3_meanfreq,channel4_rms,...,channel6_mav,channel6_meanfreq,channel7_rms,channel7_mav,channel7_meanfreq,channel8_rms,channel8_mav,channel8_meanfreq,class,Subject_id
0,0.000033,0.000028,118.382484,0.000026,0.000021,122.443532,0.000019,0.000016,134.332691,0.000022,...,0.000021,129.400216,0.000024,0.000019,104.271719,0.000024,0.000020,120.135651,0.0,1
1,0.000018,0.000014,126.786966,0.000022,0.000016,131.953434,0.000019,0.000015,108.633024,0.000026,...,0.000020,134.086278,0.000025,0.000022,122.161093,0.000013,0.000011,144.097435,0.0,1
2,0.000028,0.000022,124.113275,0.000019,0.000015,117.114314,0.000017,0.000016,121.181428,0.000020,...,0.000033,110.809832,0.000023,0.000017,126.862723,0.000023,0.000020,100.263109,0.0,1
3,0.000017,0.000015,133.870650,0.000020,0.000017,99.510328,0.000018,0.000014,101.221043,0.000017,...,0.000029,137.485110,0.000031,0.000024,109.201729,0.000018,0.000014,138.084576,1.0,1
4,0.000020,0.000016,111.433711,0.000023,0.000018,121.003882,0.000015,0.000011,145.266405,0.000015,...,0.000028,109.193018,0.000023,0.000021,123.220154,0.000024,0.000017,116.262733,1.0,1


In [36]:
final_df['class'].value_counts()

class
0.0    13583
5.0     1262
6.0     1262
4.0     1261
1.0     1252
3.0     1252
2.0     1213
7.0       69
Name: count, dtype: int64

By running this cell , i found that there is high no. o class '0' , i mean those were insanely high 
and from the dataset it is classified to be a "unmarked Data" . also the class '7' is basically non existant . which means this can cause a lot of problem to our model's predicting ability . so i decided to drop them 

In [37]:
final_dfc = final_df[(final_df['class'] != 0) & (final_df['class'] != 7)]

In [38]:

X = final_dfc.drop(columns={'class','Subject_id'})
y = final_dfc['class']
final_dfc.shape



(7502, 26)

In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

train_subjects = list(range(1, 58))   # e.g. 57 subjects for train
test_subjects = list(range(58, 73))   # remaining 15 for test

train_df = final_dfc[final_dfc['Subject_id'].isin(train_subjects)]
test_df = final_dfc[final_dfc['Subject_id'].isin(test_subjects)]

feature_cols = [col for col in final_dfc.columns if col not in ['class', 'Subject_id']]

X_train = train_df[feature_cols]
y_train = train_df['class']

X_test = test_df[feature_cols]
y_test = test_df['class']

scaler = StandardScaler()

X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)


In [40]:
model = RandomForestClassifier(random_state=12)
kmodel = KNeighborsClassifier(n_neighbors=7)
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train,y_train)
dummy_pred = dummy.predict(X_test)
dummyscore = accuracy_score(y_test, dummy_pred)
print(f'The accuracy of the DummyClassifier Model is {dummyscore*100:.2f}%')


model.fit(X_train, y_train)
predictions_rfc = model.predict(X_test)
accuracy_rfc = accuracy_score(y_test, predictions_rfc)

kmodel.fit(X_train_scaler, y_train)
kmodel_pred = kmodel.predict(X_test_scaler)
accuracy_kmodel = accuracy_score(y_test, kmodel_pred)
print(f'The accuracy of the RandomForestClassifier Model is {accuracy_rfc*100:.2f}%')
print(classification_report(y_test, predictions_rfc))

print(f'The accuracy of the KneighboursClassifier Model is {accuracy_kmodel*100:.2f}%')
print(classification_report(y_test, kmodel_pred))



The accuracy of the DummyClassifier Model is 16.66%
The accuracy of the RandomForestClassifier Model is 88.90%
              precision    recall  f1-score   support

         1.0       1.00      1.00      1.00       260
         2.0       0.91      0.93      0.92       253
         3.0       0.85      0.84      0.84       261
         4.0       0.91      0.84      0.88       261
         5.0       0.84      0.98      0.91       264
         6.0       0.84      0.74      0.79       268

    accuracy                           0.89      1567
   macro avg       0.89      0.89      0.89      1567
weighted avg       0.89      0.89      0.89      1567

The accuracy of the KneighboursClassifier Model is 84.43%
              precision    recall  f1-score   support

         1.0       0.96      0.99      0.97       260
         2.0       0.87      0.87      0.87       253
         3.0       0.75      0.77      0.76       261
         4.0       0.85      0.86      0.85       261
         5.0     

Here , i tried running this code with class '0' and '7' being present on the final_df['class']
this resulted in a low accuracy score. moreover '0' meant unmarked data and '7' is almost non-exsistent

In [41]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, predictions_rfc)
cm_df = pd.DataFrame(cm, index=sorted(y_test.unique()), columns=sorted(y_test.unique()))
print(cm_df)

     1.0  2.0  3.0  4.0  5.0  6.0
1.0  260    0    0    0    0    0
2.0    0  235    2    0    9    7
3.0    0    6  219    1    5   30
4.0    0    4    0  220   35    2
5.0    0    1    0    3  260    0
6.0    1   13   38   17    0  199


This is the feature_importance model 

In [45]:

from sklearn.datasets import make_classification

import pandas as pd


feature_names = [f"{i}" for i in X_train]

# 2. Train a Random Forest
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 3. Extract and format feature importances
importances = model.feature_importances_
df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
df = df.sort_values(by='Importance', ascending=False)

print(df)


              Feature  Importance
18       channel7_rms    0.087816
19       channel7_mav    0.080823
21       channel8_rms    0.076456
12       channel5_rms    0.071462
9        channel4_rms    0.071302
10       channel4_mav    0.069999
0        channel1_rms    0.066917
13       channel5_mav    0.065078
22       channel8_mav    0.059704
1        channel1_mav    0.051193
15       channel6_rms    0.049681
16       channel6_mav    0.046238
3        channel2_rms    0.040677
4        channel2_mav    0.038331
7        channel3_mav    0.030736
6        channel3_rms    0.029277
20  channel7_meanfreq    0.008594
2   channel1_meanfreq    0.008273
23  channel8_meanfreq    0.008218
11  channel4_meanfreq    0.008127
5   channel2_meanfreq    0.008085
17  channel6_meanfreq    0.008003
14  channel5_meanfreq    0.007585
8   channel3_meanfreq    0.007425


In [43]:
X_train.columns.to_list()

['channel1_rms',
 'channel1_mav',
 'channel1_meanfreq',
 'channel2_rms',
 'channel2_mav',
 'channel2_meanfreq',
 'channel3_rms',
 'channel3_mav',
 'channel3_meanfreq',
 'channel4_rms',
 'channel4_mav',
 'channel4_meanfreq',
 'channel5_rms',
 'channel5_mav',
 'channel5_meanfreq',
 'channel6_rms',
 'channel6_mav',
 'channel6_meanfreq',
 'channel7_rms',
 'channel7_mav',
 'channel7_meanfreq',
 'channel8_rms',
 'channel8_mav',
 'channel8_meanfreq']